# Lecture : Graph Transformers

## Lab 02 : Graph Transformers (GT with Positional Encoding) -- Exercise

### Xavier Bresson, Zhou Wei

Dwivedi, Bresson, A generalization of transformer networks to graphs, 2020   
https://arxiv.org/pdf/2012.09699.pdf


In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/07_Graph_Transformers'
    print(path_to_file)
    os.chdir(path_to_file)
    !pwd
    !pip -q install torch_geometric==2.5.1


In [ ]:
# Libraries
import math
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
from sklearn.cluster import SpectralClustering
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import (
    add_self_loops,
    degree,
    from_networkx,
    softmax,
    to_dense_adj,
    to_networkx,
)


# Visualise the eight PyG MiniGC graph classes (Guoji initialization).


In [ ]:
# Guoji's PyG MiniGC initialization (from code03_solution_pyg.ipynb)
CLASS_NAMES = [
    "cycle", "star", "wheel", "lollipop",
    "hypercube", "grid", "clique", "circular ladder",
]


def networkx_to_pyg(graph, label):
    graph = nx.convert_node_labels_to_integers(graph)
    data = from_networkx(graph)
    data.edge_index, _ = add_self_loops(
        data.edge_index, num_nodes=data.num_nodes
    )
    data.y = torch.tensor([label], dtype=torch.long)
    return data


def generate_minigc_pyg(num_graphs, min_num_v, max_num_v, seed=0):
    rng = np.random.RandomState(seed)
    dataset = []
    graphs_per_class = num_graphs // 8

    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        dataset.append(networkx_to_pyg(nx.cycle_graph(num_v), 0))
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        dataset.append(networkx_to_pyg(nx.star_graph(num_v - 1), 1))
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        dataset.append(networkx_to_pyg(nx.wheel_graph(num_v), 2))
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        path_len = int(rng.randint(2, num_v // 2))
        dataset.append(
            networkx_to_pyg(nx.lollipop_graph(num_v - path_len, path_len), 3)
        )
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        dimension = int(math.log(num_v, 2))
        dataset.append(networkx_to_pyg(nx.hypercube_graph(dimension), 4))
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        num_rows = int(rng.randint(2, num_v // 2))
        num_cols = num_v // num_rows
        dataset.append(networkx_to_pyg(nx.grid_graph([num_rows, num_cols]), 5))
    for _ in range(graphs_per_class):
        num_v = int(rng.randint(min_num_v, max_num_v))
        dataset.append(networkx_to_pyg(nx.complete_graph(num_v), 6))
    while len(dataset) < num_graphs:
        num_v = int(rng.randint(min_num_v, max_num_v))
        dataset.append(
            networkx_to_pyg(nx.circular_ladder_graph(num_v // 2), 7)
        )
    return dataset


# Visualise one example from each of the eight classes.
dataset = generate_minigc_pyg(8, 10, 20, seed=0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for c, ax in enumerate(axes.ravel()):
    graph = to_networkx(
        dataset[c], to_undirected=True, remove_self_loops=True
    )
    nx.draw(graph, ax=ax, node_size=70, width=1.0)
    ax.set_title(f"Class {c}: {CLASS_NAMES[c]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


# Exercise 1: Generate train, val and test datasets



### Question 1.1: Use Laplacian eigenvectors as positional encoding

**Instructions:**

- *Step 1:* Compute the normalized Laplacian $L = I - D^{-1/2}AD^{-1/2}$. In PyG, use `to_dense_adj(data.edge_index, max_num_nodes=data.num_nodes)[0]` for $A$ and `degree(data.edge_index[1], num_nodes=data.num_nodes)` for degrees.
- *Step 2:* Compute the eigenvalues and eigenvectors with `torch.linalg.eig()`.
- *Step 3:* Convert the eigenvalues and eigenvectors to real values with `.real`.
- *Step 4:* Sort eigenvectors by increasing eigenvalue.
- *Step 5:* Skip the trivial first eigenvector and select `pos_enc_dim` columns.


In [ ]:
# Laplacian eigenvectors as positional encoding

def LapEig_positional_encoding(data, pos_enc_dim):
    ###############################################
    # YOUR CODE STARTS
    ###############################################
    # Step 1: build the adjacency matrix and normalized Laplacian.
    Adj = ...  # dense adjacency from data.edge_index
    Dn = ...  # diagonal matrix of inverse square-root degrees
    Lap = ...  # I - Dn @ Adj @ Dn

    # Step 2: compute eigenvalues and eigenvectors.
    EigVal, EigVec = ...

    # Step 3: keep the real parts.
    EigVal, EigVec = ...

    # Step 4: sort the eigenvectors by increasing eigenvalue.
    EigVec = ...

    # Step 5: skip the trivial first eigenvector.
    EigVec = ...
    ###############################################
    # YOUR CODE ENDS
    ###############################################
    return EigVec


### Question 1.2: Add positional encodings to PyG graphs

The dataset-preparation cell below assigns node in-degree features to `data.x`. Add the Laplacian eigenvectors to each graph as `data.pos_enc`.


In [ ]:
# Add positional encoding features to graphs.
pos_enc_dim = 2

def add_positional_encoding(dataset):
    for graph in dataset:
        ###############################################
        # YOUR CODE STARTS
        ###############################################
        graph.pos_enc = ...  # use LapEig_positional_encoding
        ###############################################
        # YOUR CODE ENDS
        ###############################################
    return dataset


### Generate graph datasets

Use the same graph-family initialization as Guoji's `code03_solution_pyg.ipynb`, with a separate fixed seed for each split. The graphs include both directions of each undirected edge and one self-loop per node.


In [ ]:
# Add node in-degree features, as in Guoji's PyG initialization.
def add_node_features(dataset):
    for graph in dataset:
        dst = graph.edge_index[1]
        graph.x = degree(dst, num_nodes=graph.num_nodes).view(-1, 1).float()
    return dataset


trainset = add_node_features(generate_minigc_pyg(350, 10, 20, seed=0))
testset = add_node_features(generate_minigc_pyg(100, 10, 20, seed=1))
valset = add_node_features(generate_minigc_pyg(100, 10, 20, seed=2))
trainset = add_positional_encoding(trainset)
testset = add_positional_encoding(testset)
valset = add_positional_encoding(valset)
print(trainset[0])


### Visualize positional encoding

In [ ]:
graph = trainset[0]

# Visualize the graph and a four-way spectral clustering of its nodes.
Adj = to_dense_adj(graph.edge_index, max_num_nodes=graph.num_nodes)[0]
Adj_vis = Adj.clone()
Adj_vis.fill_diagonal_(0)
A_nx = nx.from_numpy_array(Adj_vis.numpy())
clusters = SpectralClustering(
    n_clusters=4, affinity="precomputed", random_state=0
).fit_predict(Adj_vis.numpy())

fig, ax = plt.subplots(figsize=(6, 5))
nx.draw(A_nx, ax=ax, node_color=clusters, cmap="jet", with_labels=True)
ax.set_title("Graph nodes colored by spectral clustering")
plt.show()

# Plot the two positional-encoding coordinates.
fig, ax = plt.subplots(figsize=(6, 5))
x = graph.pos_enc
ax.scatter(x[:, 0], x[:, 1], c=clusters, cmap="jet")
for i in range(graph.num_nodes):
    ax.annotate(str(i), (x[i, 0], x[i, 1]), textcoords="offset points", xytext=(1, 5))
ax.set_title("2D Laplacian positional encoding")
plt.show()


### Use PyG DataLoader to prepare and inspect a batch of graphs


In [ ]:
# PyG DataLoader batches graphs, offsets edge indices, and creates a node-to-graph `batch` vector.
batch_size = 10
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
batch_graphs = next(iter(train_loader))
batch_labels = batch_graphs.y
batch_pe = batch_graphs.pos_enc
print(batch_graphs)
print(batch_labels)
print("batch_pe:", batch_pe.size())
print("batch vector:", batch_graphs.batch.size())


# Exercise 2: Design the class of GraphTransformer networks (with positional encoding)

Node update equation:
\begin{eqnarray*}
\bar{h}^{\ell} &=&  h^{\ell} + \textrm{gMHA} (\textrm{LN}(h^{\ell})) \in \mathbb{R}^{N\times d}\\
h^{\ell+1} &=& \bar{h}^{\ell} + \textrm{MLP} (\textrm{LN}(\bar{h}^{\ell})) \in \mathbb{R}^{N\times d}\\
&&\textrm{with } \textrm{gMHA}(h)=\textrm{Concat}_{k=1}^H \left( \textrm{gHA}(h_k) \right) W_O \in \mathbb{R}^{N\times d},\ h_k\in \mathbb{R}^{N\times d'=d/H}, W_O\in \mathbb{R}^{d\times d} \\
&&\quad\quad\ \textrm{gHA}(h)=\textrm{Softmax}\left( A_G \odot \frac{QK^T}{\sqrt{d'}} \right) V \in \mathbb{R}^{N\times d'=d/H}, A_G\in \mathbb{R}^{N\times N} \textrm{ (graph adjacency matrix)}\\
&&\quad\quad\ \textrm{gHA}(h)_i= \sum_{j\in \mathcal{N}_i} \underbrace{\frac{\exp(q_i^T k_j/\sqrt{d'})}{ \sum_{j'\in\mathcal{N}_i} \exp(q_i^T k_{j'}/\sqrt{d'}) }}_{\textrm{graph attention score}_{ij}} v_j\ \textrm{ (point-wise equation)}\\
&&\quad\quad\ Q=h_k W_Q, K=h_k W_K, V=h_k W_V\in \mathbb{R}^{N\times d'=d/H}, W_Q, W_K, W_V\in \mathbb{R}^{d'\times d'}\\
h^{\ell=0} &=& \textrm{LL}_1(h_0)+\textrm{LL}_2(p_0) \in \mathbb{R}^{N\times d}\ \textrm{(input node feature and positional encoding)}\\
&&\textrm{with } p_0=\Phi_{\{2,..,K+1\}}\in \mathbb{R}^{N\times K},\ \Delta = \Phi \Lambda \Phi^T \in \mathbb{R}^{N\times N}
\end{eqnarray*}


### Implement a MLP layer for classification like the previous exercise

In [ ]:
# class MLP layer for classification
class MLP_layer(nn.Module):

    def __init__(self, input_dim, output_dim, L=2): # L = nb of hidden layers
        super(MLP_layer, self).__init__()
        list_FC_layers = [ nn.Linear( input_dim, input_dim, bias=True ) for l in range(L) ]
        list_FC_layers.append(nn.Linear( input_dim, output_dim , bias=True ))
        self.FC_layers = nn.ModuleList(list_FC_layers)
        self.L = L

    def forward(self, x):
        y = x
        for l in range(self.L):
            y = self.FC_layers[l](y)
            y = torch.relu(y)
        y = self.FC_layers[self.L](y)
        return y



### Implement graph multi-head attention from scratch with PyG `MessagePassing`


In [ ]:
# Graph multi-head attention implemented explicitly with PyG MessagePassing.
class graph_MHA_layer(MessagePassing):
    propagate_type = {"q": torch.Tensor, "k": torch.Tensor, "v": torch.Tensor}

    def __init__(self, hidden_dim, head_hidden_dim, num_heads):
        super().__init__(aggr="add", flow="source_to_target", node_dim=0)
        self.head_hidden_dim = head_hidden_dim
        self.num_heads = num_heads
        self.WQ = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WK = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WV = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)

    def message(self, q_i, k_j, v_j, index, ptr, size_i):
        # Compute scaled dot-product attention and normalize over incoming neighbors.
        score = (q_i * k_j).sum(dim=-1, keepdim=True)
        score = score / math.sqrt(self.head_hidden_dim)
        alpha = softmax(score, index=index, ptr=ptr, num_nodes=size_i)
        return alpha * v_j

    def forward(self, edge_index, h):
        num_nodes = h.size(0)
        q = self.WQ(h).view(num_nodes, self.num_heads, self.head_hidden_dim)
        k = self.WK(h).view(num_nodes, self.num_heads, self.head_hidden_dim)
        v = self.WV(h).view(num_nodes, self.num_heads, self.head_hidden_dim)
        return self.propagate(
            edge_index,
            q=q,
            k=k,
            v=v,
            size=(num_nodes, num_nodes),
        )


### Implement a Graph Transformer layer with PyG graph attention


In [ ]:
# Graph Transformer layer: attention residual followed by MLP residual.
class GraphTransformer_layer(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.0):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.dropout_mha = nn.Dropout(dropout)
        self.dropout_mlp = nn.Dropout(dropout)
        self.gMHA = graph_MHA_layer(hidden_dim, hidden_dim // num_heads, num_heads)
        self.WO = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm1 = nn.LayerNorm(hidden_dim)
        self.layer_norm2 = nn.LayerNorm(hidden_dim)
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, edge_index, h):
        h_rc = h
        h = self.layer_norm1(h)
        h_mha = self.gMHA(edge_index, h)
        h_mha = h_mha.reshape(h.size(0), self.hidden_dim)
        h_mha = self.dropout_mha(h_mha)
        h_mha = self.WO(h_mha)
        h = h_rc + h_mha

        h_rc = h
        h = self.layer_norm2(h)
        h_mlp = self.linear1(h)
        h_mlp = torch.relu(h_mlp)
        h_mlp = self.dropout_mlp(h_mlp)
        h_mlp = self.linear2(h_mlp)
        return h_rc + h_mlp


### Question 2.1: Combine the positional embedding and Graph Transformer layers

Use `nn.Linear(pos_enc_dim, hidden_dim)` to embed `data.pos_enc`. Apply the graph Transformer layers using `data.edge_index`, then use `global_mean_pool` with `data.batch` to obtain one representation per graph before classification.


In [ ]:
# Graph Transformer network with Laplacian positional encodings.
class GraphTransformer_net(nn.Module):
    def __init__(self, net_parameters):
        super().__init__()
        pos_enc_dim = net_parameters["pos_enc_dim"]
        hidden_dim = net_parameters["hidden_dim"]
        output_dim = net_parameters["output_dim"]
        num_heads = net_parameters["num_heads"]
        num_layers = net_parameters["L"]

        ###############################################
        # YOUR CODE STARTS
        ###############################################
        self.embedding_pe = ...  # map positional encodings to hidden_dim
        ###############################################
        # YOUR CODE ENDS
        ###############################################
        self.GraphTransformer_layers = nn.ModuleList([
            GraphTransformer_layer(hidden_dim, num_heads) for _ in range(num_layers)
        ])
        self.MLP_layer = MLP_layer(hidden_dim, output_dim)

    def forward(self, data, pe):
        ###############################################
        # YOUR CODE STARTS
        ###############################################
        h = ...  # embed pe to obtain node features h
        ###############################################
        # YOUR CODE ENDS
        ###############################################
        for layer in self.GraphTransformer_layers:
            h = layer(data.edge_index, h)
        graph_h = global_mean_pool(h, data.batch)
        return self.MLP_layer(graph_h)


### Question 2.2: Instantiate a Graph Transformer with positional encoding


In [ ]:
# Instantiate and inspect one network.
net_parameters = {
    "input_dim": 1,
    "pos_enc_dim": pos_enc_dim,
    "hidden_dim": 128,
    "output_dim": 8,
    "num_heads": 8,
    "L": 4,
}
net = GraphTransformer_net(net_parameters)
print(net)


def display_num_param(net):
    nb_param = sum(param.numel() for param in net.parameters())
    print("Number of parameters: {} ({:.2f} million)".format(nb_param, nb_param / 1e6))
    return nb_param / 1e6


_ = display_num_param(net)

batch_size = 10
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
batch_graphs = next(iter(train_loader))
batch_labels = batch_graphs.y
###############################################
# YOUR CODE STARTS
###############################################
batch_pe = ...  # read positional encodings from the PyG batch
###############################################
# YOUR CODE ENDS
###############################################
batch_scores = net(batch_graphs, batch_pe)
print("batch_scores:", batch_scores.size())


# Train the network

In [ ]:
def accuracy(scores, targets):
    predictions = scores.detach().argmax(dim=1)
    return (predictions == targets).float().sum().item()


def run_one_epoch(net, data_loader, train=True, loss_fc=None, optimizer=None):
    net.train() if train else net.eval()
    epoch_loss = 0.0
    epoch_acc = 0.0
    nb_data = 0

    for batch_graphs in data_loader:
        batch_labels = batch_graphs.y
        batch_pe = batch_graphs.pos_enc
        if train:
            # Laplacian eigenvectors have arbitrary signs; augment by random sign flips only in training.
            sign = 2 * torch.randint(
                low=0, high=2, size=(1, pos_enc_dim), device=batch_pe.device
            ).to(batch_pe.dtype) - 1.0
            batch_pe = batch_pe * sign

        batch_scores = net(batch_graphs, batch_pe)
        loss = loss_fc(batch_scores, batch_labels)
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_count = batch_labels.size(0)
        epoch_loss += loss.detach().item() * batch_count
        epoch_acc += accuracy(batch_scores, batch_labels)
        nb_data += batch_count

    epoch_loss /= nb_data
    epoch_acc /= nb_data
    return epoch_loss, epoch_acc


train_loader = DataLoader(trainset, batch_size=50, shuffle=True)
test_loader = DataLoader(testset, batch_size=50, shuffle=False)
val_loader = DataLoader(valset, batch_size=50, shuffle=False)

torch.manual_seed(0)
net_parameters = {
    "input_dim": 1,
    "pos_enc_dim": pos_enc_dim,
    "hidden_dim": 128,
    "output_dim": 8,
    "num_heads": 8,
    "L": 4,
}
net = GraphTransformer_net(net_parameters)
_ = display_num_param(net)
loss_fc = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.0003)

start = time.time()
for epoch in range(50):
    epoch_train_loss, epoch_train_acc = run_one_epoch(
        net, train_loader, train=True, loss_fc=loss_fc, optimizer=optimizer
    )
    with torch.no_grad():
        epoch_test_loss, epoch_test_acc = run_one_epoch(
            net, test_loader, train=False, loss_fc=loss_fc
        )
        epoch_val_loss, epoch_val_acc = run_one_epoch(
            net, val_loader, train=False, loss_fc=loss_fc
        )
    print(
        "Epoch {}, time {:.4f}, train_loss: {:.4f}, test_loss: {:.4f}, val_loss: {:.4f}".format(
            epoch, time.time() - start, epoch_train_loss, epoch_test_loss, epoch_val_loss
        )
    )
    print(
        "                       train_acc: {:.4f}, test_acc: {:.4f}, val_acc: {:.4f}".format(
            epoch_train_acc, epoch_test_acc, epoch_val_acc
        )
    )


## Compare results

Historical course references (the run above reports this notebook's metrics):

| GNN | train acc | test acc |
| -------- | ------- | ------- |
| GCN | 0.7829 | 0.7900 |
| GIN | 0.0800 | 0.1000 |
| GAT | 0.9229 | 0.9400 |
| Vanilla GT | 0.9229 | 0.9400 |
| GT (with Laplacian PE) | see training output above | see training output above |
